In [ ]:
!pip install neo4j vllm outlines transformers
!pip install protobuf==3.20.3

In [ ]:
from google.colab import drive

# Clean previous mounts
try:
    drive.flush_and_unmount()
    print("Unmount successful.")
except Exception:
    print("No active drive found. Proceeding...")

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
print("Drive ready.")

baseline: model + alpaca

In [ ]:
import os
from transformers import AutoTokenizer
from vllm import LLM
from google.colab import userdata

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["VLLM_USE_V1"] = "0"

# Model selection
#model_id = "unsloth/Llama-3.2-1B-Instruct"   
model_id = "Qwen/Qwen3-0.6B"
print(f"Preparing {model_id}...")

# Base vLLM parameters
vllm_kwargs = {
    "model": model_id,
    "trust_remote_code": True,
    "max_model_len": 2048,
    "enforce_eager": True,
    "max_num_seqs": 16,
    "max_num_batched_tokens": 4096,
    "gpu_memory_utilization": 0.75,
}


# Initialize engine
try:
    print("Initializing vLLM...")
    llm = LLM(**vllm_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
except Exception as e:
    print(f"Load failed: {e}")
    raise e

# Tokenizer setup
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model {model_id} ready.")

In [ ]:
import random
from neo4j import GraphDatabase
from google.colab import userdata


# NEO4J SETUP
print("Connecting to Neo4j...")
URI = userdata.get('NEO4J_URI')
USERNAME = userdata.get('NEO4J_USERNAME')
PASSWORD = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
print("Connected.")

def get_kg_context(driver_instance, char_class, version):
    class_idx = (char_class or "").lower().replace(" ", "-")
    query = """
    MATCH (c:Class {index: $class_idx, srd_version: $version})
    OPTIONAL MATCH (c)-[:HAS_PROFICIENCY]->(cp:Proficiency)
    OPTIONAL MATCH (c)-[:PROFICIENT_IN_SAVE]->(save:Stat)
    OPTIONAL MATCH (c)-[:USES_MAGIC_STAT]->(magic:Stat)
    OPTIONAL MATCH (c)-[:STARTING_EQUIPMENT]->(ce:Equipment)
    RETURN c.hit_die AS hit_die, c.spellcasting_type AS spell_type, magic.name AS spellcasting_ability,
           collect(DISTINCT save.name) AS saving_throws, collect(DISTINCT cp.name) AS proficiencies,
           collect(DISTINCT ce.name) AS equipment
    """
    with driver_instance.session() as session:
        result = session.run(query, class_idx=class_idx, version=version)
        record = result.single()
        if record and record["hit_die"]:
            spell_ability = f" (Ability: {record['spellcasting_ability']})" if record['spellcasting_ability'] else ""
            return (f"RULES CONTEXT ({version} Edition):\n- Hit Die: {record['hit_die']}\n"
                    f"- Saving Throws: {', '.join(record['saving_throws'])}\n"
                    f"- Spellcasting: {record['spell_type']}{spell_ability}\n"
                    f"- Class Proficiencies: {', '.join(record['proficiencies'])}\n"
                    f"- Starting Equipment: {', '.join(record['equipment'])}\n")
    return f"RULES CONTEXT: Standard D&D 5e {version} rules apply."

def get_completion_options_from_kg(driver_instance, masked_field, sheet_json, version):
    class_idx = (sheet_json.get("class") or "").lower().replace(" ", "-")
    subclass_idx = (sheet_json.get("subclass") or "").lower().replace(" ", "-").replace("'", "")
    raw_options = []

    with driver_instance.session() as session:
        query = ""
        params = {"version": version, "class_idx": class_idx, "subclass_idx": subclass_idx}

        if masked_field == "subclass" and class_idx:
            query = """
            MATCH (c:Class {index: $class_idx, srd_version: $version})-[:HAS_SUBCLASS]->(sc:Subclass)
            WITH collect(sc.name) AS valid_sc
            MATCH (bad_sc:Subclass {srd_version: $version}) WHERE NOT bad_sc.name IN valid_sc
            WITH valid_sc, bad_sc, rand() as r ORDER BY r LIMIT 5
            WITH valid_sc, collect(bad_sc.name) AS bad_list
            RETURN valid_sc + bad_list AS options
            """
        elif masked_field in ["class", "race", "background", "feats"]:
            label = "Feat" if masked_field == "feats" else masked_field.capitalize()
            query = f"MATCH (n:{label} {{srd_version: $version}}) WITH n, rand() as r ORDER BY r LIMIT 10 RETURN collect(n.name) AS options"
        elif masked_field == "spells":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_LEARN]->(s1:Spell)
            WITH collect(s1.name) as valid_spells
            OPTIONAL MATCH (sc:Subclass {index: $subclass_idx, srd_version: $version})-[:CAN_LEARN]->(s2:Spell)
            WITH valid_spells, collect(s2.name) as sub_spells
            WITH valid_spells + sub_spells as valid_spells
            MATCH (s:Spell {srd_version: $version}) WHERE s.name IN valid_spells WITH collect(s.name)[..10] AS final_valid, valid_spells
            MATCH (bad_s:Spell {srd_version: $version}) WHERE NOT bad_s.name IN valid_spells
            WITH final_valid, bad_s, rand() as r ORDER BY r LIMIT 10
            WITH final_valid, collect(bad_s.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "skills":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_CHOOSE_SKILL]->(sk1:Skill) WITH collect(sk1.name) as valid_skills
            MATCH (sk:Skill {srd_version: $version}) WHERE sk.name IN valid_skills WITH collect(sk.name) AS final_valid, valid_skills
            MATCH (bad_sk:Skill {srd_version: $version}) WHERE NOT bad_sk.name IN valid_skills
            WITH final_valid, bad_sk, rand() as r ORDER BY r LIMIT 5
            WITH final_valid, collect(bad_sk.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "weapons":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:STARTING_EQUIPMENT]->(w1:Weapon) WITH collect(w1.name) as valid_weapons
            MATCH (w:Weapon:Equipment {srd_version: $version}) WHERE w.name IN valid_weapons WITH collect(w.name) AS final_valid, valid_weapons
            MATCH (bad_w:Weapon:Equipment {srd_version: $version}) WHERE NOT bad_w.name IN valid_weapons
            WITH final_valid, bad_w, rand() as r ORDER BY r LIMIT 8
            WITH final_valid, collect(bad_w.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """

        if query:
            result = session.run(query, **params)
            record = result.single()
            if record and record["options"]:
                raw_options = list(set([opt for opt in record["options"] if opt]))
                random.shuffle(raw_options)

    options_str = ""
    if raw_options:
        options_str = f"Options to choose from:\n- " + "\n- ".join(raw_options)
    return options_str, raw_options

In [ ]:
import os
import json
import random
from tqdm import tqdm
from google.colab import userdata
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

os.environ["VLLM_USE_V1"] = "0"


# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TEST_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_test.json")
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, "inference_results_base_alpaca")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

safe_model_name = model_id.split("/")[-1]
output_file = os.path.join(OUTPUT_RESULTS_DIR, f"predictions_{safe_model_name}_Base_Alpaca.json")

# ALPACA TEMPLATE
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# DATA LOADING
with open(TEST_DATA_PATH, 'r') as f:
    test_data = json.load(f)

results = []
if os.path.exists(output_file):
    try:
        with open(output_file, 'r') as f: results = json.load(f)
        print(f"Resuming from {len(results)}")
    except json.JSONDecodeError: pass

data_slice = test_data[len(results):]

# BATCH PREPARATION
if len(data_slice) > 0:
    all_prompts = []
    all_sampling_params = []

    for sample in tqdm(data_slice, desc="Preparing batch"):
        task_type = sample.get("task_type", "generation")
        version = sample.get("version", "2014")
        raw_prompt = sample.get("llm_prompt", "")

        # Parse prompt/input
        if "Input Data:" in raw_prompt:
            parts = raw_prompt.split("Input Data:")
            original_instruction = parts[0].strip()
            input_str = parts[1].strip()
            try:
                sheet_json = json.loads(input_str)
            except json.JSONDecodeError:
                sheet_json = {}
        else:
            original_instruction = raw_prompt
            input_str = "{}"
            sheet_json = {}

        # Generation params
        base_sampling_kwargs = {
            "temperature": 0.2,
            "max_tokens": 1024,
            "repetition_penalty": 1.1,
            "stop": [
                "###", "### Instruction:", "### Input:", "### Response:",
                "<|im_end|>", "<|im_start|>", "<|endoftext|>",
                "<|eot_id|>", "<|eom_id|>", "<|start_header_id|>", "<|end_header_id|>"
            ]
        }
        sampling_params = SamplingParams(**base_sampling_kwargs)

        # Task logic
        if task_type == "generation":
            context = get_kg_context(driver, sheet_json.get("class", ""), version)
            user_prompt = f"{original_instruction}\n\n[Rules Context]\n{context}"

        elif task_type == "completion":
            target_keys = [k for k, v in sheet_json.items() if v is None or v == "null"]
            masked_field = sample.get("masked_field") or (target_keys[0] if target_keys else "class")
            options_str, raw_options_list = get_completion_options_from_kg(driver, masked_field, sheet_json, version)
            options_part = f"\n\n{options_str}" if options_str else ""
            user_prompt = f"{original_instruction}{options_part}"

            # Structured output
            if masked_field in ["hp", "ac", "level"]:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(regex=r"^[0-9]+$")
                )
            elif masked_field in ["class", "subclass", "race", "background", "alignment"] and raw_options_list:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(choice=raw_options_list)
                )

        elif task_type == "refusal":
            user_prompt = original_instruction.replace("Generate a complete D&D 5e character sheet based on the provided attributes.", "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules.")

        full_prompt = alpaca_prompt.format(user_prompt, input_str)
        all_prompts.append(full_prompt)
        all_sampling_params.append(sampling_params)

    # INFERENCE
    CHUNK_SIZE = 50
    for i in range(0, len(all_prompts), CHUNK_SIZE):
        chunk_prompts = all_prompts[i : i + CHUNK_SIZE]
        chunk_params = all_sampling_params[i : i + CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}...")

        chunk_outputs = llm.generate(chunk_prompts, sampling_params=chunk_params, use_tqdm=True)

        for idx, output in enumerate(chunk_outputs):
            global_idx = i + idx
            results.append({
                "task_type": data_slice[global_idx].get("task_type"),
                "input_prompt": all_prompts[global_idx],
                "generated_response": output.outputs[0].text.strip(),
                "expected_output": data_slice[global_idx].get("expected_output")
            })

        # Checkpoint save
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)

    print("Inference complete.")
driver.close()

base + chat template

In [ ]:
import os
import json
import random
from tqdm import tqdm
from neo4j import GraphDatabase
from google.colab import userdata
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

os.environ["VLLM_USE_V1"] = "0"


# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TEST_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_test.json")
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, "inference_results_base_chat")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

safe_model_name = model_id.split("/")[-1]
output_file = os.path.join(OUTPUT_RESULTS_DIR, f"predictions_{safe_model_name}_Base_Chat.json")

# DATA LOADING
with open(TEST_DATA_PATH, 'r') as f:
    test_data = json.load(f)

results = []
if os.path.exists(output_file):
    try:
        with open(output_file, 'r') as f: results = json.load(f)
        print(f"Resuming from {len(results)}")
    except json.JSONDecodeError: pass

data_slice = test_data[len(results):]

# BATCH PREPARATION
if len(data_slice) > 0:
    all_prompts = []
    all_sampling_params = []

    for sample in tqdm(data_slice, desc="Preparing chat batch"):
        task_type = sample.get("task_type", "generation")
        version = sample.get("version", "2014")
        raw_prompt = sample.get("llm_prompt", "")

        # Parse prompt/input
        if "Input Data:" in raw_prompt:
            parts = raw_prompt.split("Input Data:")
            original_instruction = parts[0].strip()
            input_str = parts[1].strip()
            try:
                sheet_json = json.loads(input_str)
            except json.JSONDecodeError:
                sheet_json = {}
        else:
            original_instruction = raw_prompt
            input_str = "{}"
            sheet_json = {}

        # Sampling settings
        base_sampling_kwargs = {
            "temperature": 0.2,
            "max_tokens": 1024,
            "repetition_penalty": 1.1,
            "stop": [
                "<|im_end|>", "<|im_start|>", "<|endoftext|>",
                "<|eot_id|>", "<|eom_id|>", "<|start_header_id|>", "<|end_header_id|>",
                "user\n", "assistant\n"
            ]
        }
        sampling_params = SamplingParams(**base_sampling_kwargs)

        # Task Logic
        if task_type == "generation":
            context = get_kg_context(driver, sheet_json.get("class", ""), version)
            user_prompt = f"{original_instruction}\n\n[Rules Context]\n{context}\n\nInput Data:\n{input_str}"

        elif task_type == "completion":
            target_keys = [k for k, v in sheet_json.items() if v is None or v == "null"]
            masked_field = sample.get("masked_field") or (target_keys[0] if target_keys else "class")
            options_str, raw_options_list = get_completion_options_from_kg(driver, masked_field, sheet_json, version)
            options_part = f"\n\n{options_str}" if options_str else ""
            user_prompt = f"{original_instruction}{options_part}\n\nInput Data:\n{input_str}"

            if masked_field in ["hp", "ac", "level"]:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(regex=r"^[0-9]+$")
                )
            elif masked_field in ["class", "subclass", "race", "background", "alignment"] and raw_options_list:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(choice=raw_options_list)
                )

        elif task_type == "refusal":
            refusal_instr = original_instruction.replace("Generate a complete D&D 5e character sheet based on the provided attributes.", "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules.")
            user_prompt = f"{refusal_instr}\n\nInput Data:\n{input_str}"

        # Chat template application
        messages = [{"role": "user", "content": user_prompt}]
        template_kwargs = {"tokenize": False, "add_generation_prompt": True}

        if "qwen" in safe_model_name.lower():
            try:
                full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs, enable_thinking=False)
            except TypeError:
                full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs)
        else:
            full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs)

        all_prompts.append(full_prompt)
        all_sampling_params.append(sampling_params)

    # NFERENCE
    CHUNK_SIZE = 50
    for i in range(0, len(all_prompts), CHUNK_SIZE):
        chunk_prompts = all_prompts[i : i + CHUNK_SIZE]
        chunk_params = all_sampling_params[i : i + CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}...")

        chunk_outputs = llm.generate(chunk_prompts, sampling_params=chunk_params, use_tqdm=True)

        for idx, output in enumerate(chunk_outputs):
            global_idx = i + idx
            results.append({
                "task_type": data_slice[global_idx].get("task_type"),
                "input_prompt": all_prompts[global_idx],
                "generated_response": output.outputs[0].text.strip(),
                "expected_output": data_slice[global_idx].get("expected_output")
            })

        with open(output_file, 'w') as f: json.dump(results, f, indent=2)

    print("Inference complete.")
driver.close()

fine tuning Alpaca

In [ ]:
!pip install --no-deps unsloth bitsandbytes accelerate xformers peft trl tokenizers
!pip install --no-deps "transformers>=4.51.0"
!pip install sentencepiece protobuf huggingface_hub hf_transfer
!pip install unsloth_zoo

In [ ]:
import os
import json
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TRAIN_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_train.json")
OUTPUT_MODELS_DIR = os.path.join(BASE_DIR, "fine_tuned_models_alpaca")
os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)

# #model_id = "unsloth/Llama-3.2-1B-Instruct"
model_id = "Qwen/Qwen3-0.6B"
safe_name = model_id.split("/")[-1]
save_merged_path = os.path.join(OUTPUT_MODELS_DIR, f"{safe_name}_Alpaca_Merged_16bit")

# ALPACA TEMPLATE
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts_alpaca(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    task_types   = examples.get("task_type", [""] * len(instructions))
    texts = []

    for instr, inp, out, task in zip(instructions, inputs, outputs, task_types):
        # Refusal prompt
        if task == "refusal" or "refuse" in str(out).lower():
            instr = "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules."

        input_str = json.dumps(inp, indent=2) if isinstance(inp, (dict, list)) else str(inp)
        output_str = json.dumps(out, indent=2) if isinstance(out, (dict, list)) else str(out)

        text = alpaca_prompt.format(instr, input_str, output_str) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

#  LOAD MODEL & TOKENIZER
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id, 
    max_seq_length=2048, 
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model, 
    r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_alpha=16, 
    lora_dropout=0, 
    bias="none", 
    use_gradient_checkpointing="unsloth"
)

# DATASET PREP
dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train").map(format_prompts_alpaca, batched=True)

# TRAINING
trainer = SFTTrainer(
    model=model, 
    tokenizer=tokenizer, 
    train_dataset=dataset, 
    dataset_text_field="text", 
    max_seq_length=2048, 
    packing=False, 
    args=TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=4, 
        num_train_epochs=1, 
        learning_rate=2e-4, 
        fp16=not is_bfloat16_supported(), 
        bf16=is_bfloat16_supported(), 
        optim="adamw_8bit", 
        output_dir="outputs"
    )
)
trainer.train()

model.save_pretrained_merged(save_merged_path, tokenizer, save_method="merged_16bit")
print(f"Model saved to: {save_merged_path}")

inferenza alpaca

In [ ]:
import os
import sys

# vLLM environment 
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM

# Fine-tuned model path
model_id = "/content/drive/MyDrive/DnD_Project_Data_NLP/fine_tuned_models_alpaca/Qwen3-0.6B_Alpaca_Merged_16bit"
# #model_id = "/content/drive/MyDrive/DnD_Project_Data_NLP/fine_tuned_models_alpaca/Llama-3.2-1B-Instruct_Alpaca_Merged_16bit"

# Check config and load
if not os.path.exists(os.path.join(model_id, "config.json")):
    print(f"Error: config.json not found in {model_id}")
else:
    print("Config found. Initializing vLLM...")

    llm = LLM(
        model=model_id,
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
        disable_custom_all_reduce=True,
        trust_remote_code=True,
        max_num_seqs=16
    )

    # Tokenizer setup
    tokenizer = llm.get_tokenizer()
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("vLLM engine loaded for FT inference.")

In [ ]:
import random
from neo4j import GraphDatabase
from google.colab import userdata

# Neo4j connection
URI = userdata.get('NEO4J_URI')
USERNAME = userdata.get('NEO4J_USERNAME')
PASSWORD = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("Neo4j RAG connection successful.")

def get_kg_context(driver, char_class, version):
    class_idx = (char_class or "").lower().replace(" ", "-")
    query = """
    MATCH (c:Class {index: $class_idx, srd_version: $version})
    OPTIONAL MATCH (c)-[:HAS_PROFICIENCY]->(cp:Proficiency)
    OPTIONAL MATCH (c)-[:PROFICIENT_IN_SAVE]->(save:Stat)
    OPTIONAL MATCH (c)-[:USES_MAGIC_STAT]->(magic:Stat)
    OPTIONAL MATCH (c)-[:STARTING_EQUIPMENT]->(ce:Equipment)
    RETURN c.hit_die AS hit_die, c.spellcasting_type AS spell_type, magic.name AS spellcasting_ability,
           collect(DISTINCT save.name) AS saving_throws, collect(DISTINCT cp.name) AS proficiencies,
           collect(DISTINCT ce.name) AS equipment
    """
    with driver.session() as session:
        result = session.run(query, class_idx=class_idx, version=version)
        record = result.single()
        if record and record["hit_die"]:
            spell_ability = f" (Ability: {record['spellcasting_ability']})" if record['spellcasting_ability'] else ""
            return (f"RULES CONTEXT ({version} Edition):\n- Hit Die: {record['hit_die']}\n"
                    f"- Saving Throws: {', '.join(record['saving_throws'])}\n"
                    f"- Spellcasting: {record['spell_type']}{spell_ability}\n"
                    f"- Class Proficiencies: {', '.join(record['proficiencies'])}\n"
                    f"- Starting Equipment: {', '.join(record['equipment'])}\n")
    return f"RULES CONTEXT: Standard D&D 5e {version} rules apply."

def get_completion_options_from_kg(driver, masked_field, sheet_json, version):
    class_idx = (sheet_json.get("class") or "").lower().replace(" ", "-")
    subclass_idx = (sheet_json.get("subclass") or "").lower().replace(" ", "-").replace("'", "")
    raw_options = []

    with driver.session() as session:
        query = ""
        params = {"version": version, "class_idx": class_idx, "subclass_idx": subclass_idx}

        if masked_field == "subclass" and class_idx:
            query = """
            MATCH (c:Class {index: $class_idx, srd_version: $version})-[:HAS_SUBCLASS]->(sc:Subclass)
            WITH collect(sc.name) AS valid_sc
            MATCH (bad_sc:Subclass {srd_version: $version}) WHERE NOT bad_sc.name IN valid_sc
            WITH valid_sc, bad_sc, rand() as r ORDER BY r LIMIT 5
            WITH valid_sc, collect(bad_sc.name) AS bad_list
            RETURN valid_sc + bad_list AS options
            """
        elif masked_field in ["class", "race", "background", "feats"]:
            label = "Feat" if masked_field == "feats" else masked_field.capitalize()
            query = f"MATCH (n:{label} {{srd_version: $version}}) WITH n, rand() as r ORDER BY r LIMIT 10 RETURN collect(n.name) AS options"
        elif masked_field == "spells":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_LEARN]->(s1:Spell)
            WITH collect(s1.name) as valid_spells
            OPTIONAL MATCH (sc:Subclass {index: $subclass_idx, srd_version: $version})-[:CAN_LEARN]->(s2:Spell)
            WITH valid_spells, collect(s2.name) as sub_spells
            WITH valid_spells + sub_spells as valid_spells
            MATCH (s:Spell {srd_version: $version}) WHERE s.name IN valid_spells WITH collect(s.name)[..10] AS final_valid, valid_spells
            MATCH (bad_s:Spell {srd_version: $version}) WHERE NOT bad_s.name IN valid_spells
            WITH final_valid, bad_s, rand() as r ORDER BY r LIMIT 10
            WITH final_valid, collect(bad_s.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "skills":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_CHOOSE_SKILL]->(sk1:Skill) WITH collect(sk1.name) as valid_skills
            MATCH (sk:Skill {srd_version: $version}) WHERE sk.name IN valid_skills WITH collect(sk.name) AS final_valid, valid_skills
            MATCH (bad_sk:Skill {srd_version: $version}) WHERE NOT bad_sk.name IN valid_skills
            WITH final_valid, bad_sk, rand() as r ORDER BY r LIMIT 5
            WITH final_valid, collect(bad_sk.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "weapons":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:STARTING_EQUIPMENT]->(w1:Weapon) WITH collect(w1.name) as valid_weapons
            MATCH (w:Weapon:Equipment {srd_version: $version}) WHERE w.name IN valid_weapons WITH collect(w.name) AS final_valid, valid_weapons
            MATCH (bad_w:Weapon:Equipment {srd_version: $version}) WHERE NOT bad_w.name IN valid_weapons
            WITH final_valid, bad_w, rand() as r ORDER BY r LIMIT 8
            WITH final_valid, collect(bad_w.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """

        if query:
            result = session.run(query, **params)
            record = result.single()
            if record and record["options"]:
                raw_options = list(set([opt for opt in record["options"] if opt]))
                random.shuffle(raw_options)

    options_str = ""
    if raw_options:
        options_str = f"Options to choose from:\n- " + "\n- ".join(raw_options)

    return options_str, raw_options

In [ ]:
import os
import json
from tqdm import tqdm
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams

os.environ["VLLM_USE_V1"] = "0"

# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TEST_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_test.json")
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, "inference_results_alpaca")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

safe_model_name = model_id.split("/")[-1]
output_file = os.path.join(OUTPUT_RESULTS_DIR, f"predictions_{safe_model_name}.json")

# ALPACA TEMPLATE
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# LOAD DATA
with open(TEST_DATA_PATH, 'r') as f:
    test_data = json.load(f)

results = []
if os.path.exists(output_file):
    try:
        with open(output_file, 'r') as f: results = json.load(f)
        print(f"Resuming from {len(results)}")
    except json.JSONDecodeError: pass

data_slice = test_data[len(results):]

# BATCH 
if len(data_slice) > 0:
    all_prompts = []
    all_sampling_params = []

    for sample in tqdm(data_slice, desc="Preparing FT batch"):
        task_type = sample.get("task_type", "generation")
        version = sample.get("version", "2014")
        raw_prompt = sample.get("llm_prompt", "")

        # Parse prompt/input
        if "Input Data:" in raw_prompt:
            parts = raw_prompt.split("Input Data:")
            original_instruction = parts[0].strip()
            input_str = parts[1].strip()
            try: sheet_json = json.loads(input_str)
            except: sheet_json = {}
        else:
            original_instruction = raw_prompt
            input_str = "{}"
            sheet_json = {}

        # Generation params
        base_sampling_kwargs = {
            "temperature": 0.2,
            "max_tokens": 1024,
            "repetition_penalty": 1.1,
            "stop": [
                "###", "### Instruction:", "### Input:", "### Response:",
                "<|im_end|>", "<|im_start|>", "<|endoftext|>",
                "<|eot_id|>", "<|eom_id|>", "<|start_header_id|>", "<|end_header_id|>"
            ]
        }
        sampling_params = SamplingParams(**base_sampling_kwargs)

        # Task Logic
        if task_type == "generation":
            context = get_kg_context(driver, sheet_json.get("class", ""), version)
            user_prompt = f"{original_instruction}\n\n[Rules Context]\n{context}"

        elif task_type == "completion":
            target_keys = [k for k, v in sheet_json.items() if v is None or v == "null"]
            masked_field = sample.get("masked_field") or (target_keys[0] if target_keys else "class")
            options_str, raw_options_list = get_completion_options_from_kg(driver, masked_field, sheet_json, version)
            options_part = f"\n\n{options_str}" if options_str else ""
            user_prompt = f"{original_instruction}{options_part}"

            if masked_field in ["hp", "ac", "level"]:
                sampling_params = SamplingParams(**base_sampling_kwargs, structured_outputs=StructuredOutputsParams(regex=r"^[0-9]+$"))
            elif masked_field in ["class", "subclass", "race", "background", "alignment"] and raw_options_list:
                sampling_params = SamplingParams(**base_sampling_kwargs, structured_outputs=StructuredOutputsParams(choice=raw_options_list))

        elif task_type == "refusal":
            user_prompt = original_instruction.replace("Generate a complete D&D 5e character sheet based on the provided attributes.", "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules.")

        full_prompt = alpaca_prompt.format(user_prompt, input_str)
        all_prompts.append(full_prompt)
        all_sampling_params.append(sampling_params)

    # INFERENCE
    CHUNK_SIZE = 50
    for i in range(0, len(all_prompts), CHUNK_SIZE):
        chunk_prompts = all_prompts[i : i + CHUNK_SIZE]
        chunk_params = all_sampling_params[i : i + CHUNK_SIZE]
        print(f"Processing block {i//CHUNK_SIZE + 1}...")

        chunk_outputs = llm.generate(chunk_prompts, sampling_params=chunk_params, use_tqdm=True)

        for idx, output in enumerate(chunk_outputs):
            global_idx = i + idx
            results.append({
                "task_type": data_slice[global_idx].get("task_type"),
                "input_prompt": all_prompts[global_idx],
                "generated_response": output.outputs[0].text.strip(),
                "expected_output": data_slice[global_idx].get("expected_output")
            })

        with open(output_file, 'w') as f: json.dump(results, f, indent=2)

    print("FT Inference complete.")

finetuning + chat template

In [ ]:
!pip install --no-deps unsloth bitsandbytes accelerate xformers peft trl tokenizers
!pip install --no-deps "transformers>=4.51.0"
!pip install sentencepiece protobuf huggingface_hub hf_transfer
!pip install unsloth_zoo

In [ ]:
import os
import json
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TRAIN_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_train.json")
OUTPUT_MODELS_DIR = os.path.join(BASE_DIR, "fine_tuned_models_chat")
os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)

# #model_id = "unsloth/Llama-3.2-1B-Instruct"
model_id = "Qwen/Qwen3-0.6B"
safe_name = model_id.split("/")[-1]
save_merged_path = os.path.join(OUTPUT_MODELS_DIR, f"{safe_name}_Chat_Merged_16bit")

def format_prompts_chat(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    task_types   = examples.get("task_type", [""] * len(instructions))
    texts = []

    template_kwargs = {"tokenize": False}
    if "qwen" in model_id.lower():
        template_kwargs["enable_thinking"] = False

    for instr, inp, out, task in zip(instructions, inputs, outputs, task_types):
        # Refusal fix
        if task == "refusal" or "refuse" in str(out).lower():
            instr = instr.replace(
                "Generate a complete D&D 5e character sheet based on the provided attributes.",
                "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules."
            )

        input_str = json.dumps(inp, indent=2) if isinstance(inp, (dict, list)) else str(inp)
        output_str = json.dumps(out, indent=2) if isinstance(out, (dict, list)) else str(out)

        # Merge instruction and input for Chat format
        user_prompt = f"{instr}\n\nInput Data:\n{input_str}"
        messages = [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": output_str}
        ]

        try:
            text = tokenizer.apply_chat_template(messages, **template_kwargs)
        except TypeError:
            text = tokenizer.apply_chat_template(messages, tokenize=False)

        texts.append(text)
    return { "text" : texts }

# LOAD MODEL 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id, 
    max_seq_length=2048, 
    load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model, 
    r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_alpha=16, 
    lora_dropout=0, 
    bias="none", 
    use_gradient_checkpointing="unsloth"
)

# DATASET & TRAINING
dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train").map(format_prompts_chat, batched=True)

trainer = SFTTrainer(
    model=model, 
    tokenizer=tokenizer, 
    train_dataset=dataset, 
    dataset_text_field="text", 
    max_seq_length=2048, 
    packing=False, 
    args=TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=4, 
        num_train_epochs=1, 
        learning_rate=2e-4, 
        fp16=not is_bfloat16_supported(), 
        bf16=is_bfloat16_supported(), 
        optim="adamw_8bit", 
        output_dir="outputs"
    )
)
trainer.train()

# SAVE 
model.save_pretrained_merged(save_merged_path, tokenizer, save_method="merged_16bit")
print(f"Chat model saved to: {save_merged_path}")

In [ ]:
import os
import sys

os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM

# Fine-tuned model path
#model_id = "/content/drive/MyDrive/DnD_Project_Data_NLP/fine_tuned_models_chat/Qwen3-0.6B_Chat_Merged_16bit"
model_id = "/content/drive/MyDrive/DnD_Project_Data_NLP/fine_tuned_models_chat/Llama-3.2-1B-Instruct_Chat_Merged_16bit"

# Check config and load
if not os.path.exists(os.path.join(model_id, "config.json")):
    print(f"Error: config.json not found in {model_id}")
else:
    print("Config found. Initializing vLLM...")

    llm = LLM(
        model=model_id,
        max_model_len=2048,
        gpu_memory_utilization=0.75,
        enforce_eager=True,
        disable_custom_all_reduce=True,
        trust_remote_code=True,
        max_num_seqs=16
    )

    # Tokenizer setup
    tokenizer = llm.get_tokenizer()
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("vLLM engine loaded for FT Chat inference.")

inferenza chat templte post finetuning

In [ ]:
import random
from neo4j import GraphDatabase
from google.colab import userdata

# Neo4j setup
URI = userdata.get('NEO4J_URI')
USERNAME = userdata.get('NEO4J_USERNAME')
PASSWORD = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("Neo4j connection successful.")

def get_kg_context(driver, char_class, version):
    class_idx = (char_class or "").lower().replace(" ", "-")
    query = """
    MATCH (c:Class {index: $class_idx, srd_version: $version})
    OPTIONAL MATCH (c)-[:HAS_PROFICIENCY]->(cp:Proficiency)
    OPTIONAL MATCH (c)-[:PROFICIENT_IN_SAVE]->(save:Stat)
    OPTIONAL MATCH (c)-[:USES_MAGIC_STAT]->(magic:Stat)
    OPTIONAL MATCH (c)-[:STARTING_EQUIPMENT]->(ce:Equipment)
    RETURN c.hit_die AS hit_die, c.spellcasting_type AS spell_type, magic.name AS spellcasting_ability,
           collect(DISTINCT save.name) AS saving_throws, collect(DISTINCT cp.name) AS proficiencies,
           collect(DISTINCT ce.name) AS equipment
    """
    with driver.session() as session:
        result = session.run(query, class_idx=class_idx, version=version)
        record = result.single()
        if record and record["hit_die"]:
            spell_ability = f" (Ability: {record['spellcasting_ability']})" if record['spellcasting_ability'] else ""
            return (f"RULES CONTEXT ({version} Edition):\n- Hit Die: {record['hit_die']}\n"
                    f"- Saving Throws: {', '.join(record['saving_throws'])}\n"
                    f"- Spellcasting: {record['spell_type']}{spell_ability}\n"
                    f"- Class Proficiencies: {', '.join(record['proficiencies'])}\n"
                    f"- Starting Equipment: {', '.join(record['equipment'])}\n")
    return f"RULES CONTEXT: Standard D&D 5e {version} rules apply."

def get_completion_options_from_kg(driver, masked_field, sheet_json, version):
    class_idx = (sheet_json.get("class") or "").lower().replace(" ", "-")
    subclass_idx = (sheet_json.get("subclass") or "").lower().replace(" ", "-").replace("'", "")
    raw_options = []

    with driver.session() as session:
        query = ""
        params = {"version": version, "class_idx": class_idx, "subclass_idx": subclass_idx}

        if masked_field == "subclass" and class_idx:
            query = """
            MATCH (c:Class {index: $class_idx, srd_version: $version})-[:HAS_SUBCLASS]->(sc:Subclass)
            WITH collect(sc.name) AS valid_sc
            MATCH (bad_sc:Subclass {srd_version: $version}) WHERE NOT bad_sc.name IN valid_sc
            WITH valid_sc, bad_sc, rand() as r ORDER BY r LIMIT 5
            WITH valid_sc, collect(bad_sc.name) AS bad_list
            RETURN valid_sc + bad_list AS options
            """
        elif masked_field in ["class", "race", "background", "feats"]:
            label = "Feat" if masked_field == "feats" else masked_field.capitalize()
            query = f"MATCH (n:{label} {{srd_version: $version}}) WITH n, rand() as r ORDER BY r LIMIT 10 RETURN collect(n.name) AS options"
        elif masked_field == "spells":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_LEARN]->(s1:Spell)
            WITH collect(s1.name) as valid_spells
            OPTIONAL MATCH (sc:Subclass {index: $subclass_idx, srd_version: $version})-[:CAN_LEARN]->(s2:Spell)
            WITH valid_spells, collect(s2.name) as sub_spells
            WITH valid_spells + sub_spells as valid_spells
            MATCH (s:Spell {srd_version: $version}) WHERE s.name IN valid_spells WITH collect(s.name)[..10] AS final_valid, valid_spells
            MATCH (bad_s:Spell {srd_version: $version}) WHERE NOT bad_s.name IN valid_spells
            WITH final_valid, bad_s, rand() as r ORDER BY r LIMIT 10
            WITH final_valid, collect(bad_s.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "skills":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:CAN_CHOOSE_SKILL]->(sk1:Skill) WITH collect(sk1.name) as valid_skills
            MATCH (sk:Skill {srd_version: $version}) WHERE sk.name IN valid_skills WITH collect(sk.name) AS final_valid, valid_skills
            MATCH (bad_sk:Skill {srd_version: $version}) WHERE NOT bad_sk.name IN valid_skills
            WITH final_valid, bad_sk, rand() as r ORDER BY r LIMIT 5
            WITH final_valid, collect(bad_sk.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """
        elif masked_field == "weapons":
            query = """
            OPTIONAL MATCH (c:Class {index: $class_idx, srd_version: $version})-[:STARTING_EQUIPMENT]->(w1:Weapon) WITH collect(w1.name) as valid_weapons
            MATCH (w:Weapon:Equipment {srd_version: $version}) WHERE w.name IN valid_weapons WITH collect(w.name) AS final_valid, valid_weapons
            MATCH (bad_w:Weapon:Equipment {srd_version: $version}) WHERE NOT bad_w.name IN valid_weapons
            WITH final_valid, bad_w, rand() as r ORDER BY r LIMIT 8
            WITH final_valid, collect(bad_w.name) AS bad_list
            RETURN final_valid + bad_list AS options
            """

        if query:
            result = session.run(query, **params)
            record = result.single()
            if record and record["options"]:
                raw_options = list(set([opt for opt in record["options"] if opt]))
                random.shuffle(raw_options)

    options_str = ""
    if raw_options:
        options_str = "Options to choose from:\n- " + "\n- ".join(raw_options)

    return options_str, raw_options

In [ ]:
import os
import json
import random
from tqdm import tqdm
from neo4j import GraphDatabase
from google.colab import userdata
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

# vLLM anti-crash fix
os.environ["VLLM_USE_V1"] = "0"


# PATH 
BASE_DIR = "/content/drive/MyDrive/DnD_Project_Data_NLP"
TEST_DATA_PATH = os.path.join(BASE_DIR, "processed_dataset", "dnd_test.json")
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, "inference_results_chat")
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

safe_model_name = model_id.split("/")[-1]
output_file = os.path.join(OUTPUT_RESULTS_DIR, f"predictions_{safe_model_name}_Base_Chat.json")

# DATA LOADING
with open(TEST_DATA_PATH, 'r') as f:
    test_data = json.load(f)

results = []
if os.path.exists(output_file):
    try:
        with open(output_file, 'r') as f: results = json.load(f)
        print(f"Resuming from {len(results)}")
    except json.JSONDecodeError: pass

data_slice = test_data[len(results):]

# BATCH 
if len(data_slice) > 0:
    all_prompts = []
    all_sampling_params = []

    for sample in tqdm(data_slice, desc="Preparing batch"):
        task_type = sample.get("task_type", "generation")
        version = sample.get("version", "2014")
        raw_prompt = sample.get("llm_prompt", "")

        # Parse prompt/input
        if "Input Data:" in raw_prompt:
            parts = raw_prompt.split("Input Data:")
            original_instruction = parts[0].strip()
            input_str = parts[1].strip()
            try:
                sheet_json = json.loads(input_str)
            except json.JSONDecodeError:
                sheet_json = {}
        else:
            original_instruction = raw_prompt
            input_str = "{}"
            sheet_json = {}

        # Sampling settings
        base_sampling_kwargs = {
            "temperature": 0.2,
            "max_tokens": 1024,
            "repetition_penalty": 1.1,
            "stop": [
                "<|im_end|>", "<|im_start|>", "<|endoftext|>",
                "<|eot_id|>", "<|eom_id|>", "<|start_header_id|>", "<|end_header_id|>",
                "user\n", "assistant\n"
            ]
        }
        sampling_params = SamplingParams(**base_sampling_kwargs)

        # Task Logic
        if task_type == "generation":
            context = get_kg_context(driver, sheet_json.get("class", ""), version)
            user_prompt = f"{original_instruction}\n\n[Rules Context]\n{context}\n\nInput Data:\n{input_str}"

        elif task_type == "completion":
            target_keys = [k for k, v in sheet_json.items() if v is None or v == "null"]
            masked_field = sample.get("masked_field") or (target_keys[0] if target_keys else "class")
            options_str, raw_options_list = get_completion_options_from_kg(driver, masked_field, sheet_json, version)
            options_part = f"\n\n{options_str}" if options_str else ""
            user_prompt = f"{original_instruction}{options_part}\n\nInput Data:\n{input_str}"

            if masked_field in ["hp", "ac", "level"]:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(regex=r"^[0-9]+$")
                )
            elif masked_field in ["class", "subclass", "race", "background", "alignment"] and raw_options_list:
                sampling_params = SamplingParams(
                    **base_sampling_kwargs,
                    structured_outputs=StructuredOutputsParams(choice=raw_options_list)
                )

        elif task_type == "refusal":
            ref_instr = original_instruction.replace("Generate a complete D&D 5e character sheet based on the provided attributes.", "Complete this D&D 5e character sheet. Replace the NULL value(s) with correct value(s) consistent with the rules.")
            user_prompt = f"{ref_instr}\n\nInput Data:\n{input_str}"

        # Apply Chat Template
        messages = [{"role": "user", "content": user_prompt}]
        template_kwargs = {"tokenize": False, "add_generation_prompt": True}

        if "qwen" in safe_model_name.lower():
            try:
                full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs, enable_thinking=False)
            except TypeError:
                full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs)
        else:
            full_prompt = tokenizer.apply_chat_template(messages, **template_kwargs)

        all_prompts.append(full_prompt)
        all_sampling_params.append(sampling_params)

    # INFERENCE
    print(f"Starting inference on {len(all_prompts)} prompts...")
    CHUNK_SIZE = 50

    for i in range(0, len(all_prompts), CHUNK_SIZE):
        chunk_prompts = all_prompts[i : i + CHUNK_SIZE]
        chunk_params = all_sampling_params[i : i + CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}...")

        chunk_outputs = llm.generate(chunk_prompts, sampling_params=chunk_params, use_tqdm=True)

        for idx, output in enumerate(chunk_outputs):
            global_idx = i + idx
            results.append({
                "task_type": data_slice[global_idx].get("task_type"),
                "input_prompt": all_prompts[global_idx],
                "generated_response": output.outputs[0].text.strip(),
                "expected_output": data_slice[global_idx].get("expected_output")
            })

        with open(output_file, 'w') as f: json.dump(results, f, indent=2)

    print("Inference complete.")
driver.close()